# The following explores the need and motivation for our project through reduction of training data

In [17]:
import pandas as pd
import numpy as np
import itertools
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score
import importlib
import my_lstm
importlib.reload(my_lstm)

from my_lstm import build_lstm_model, create_sequences, expanding_window_lstm_forecast2

In [2]:
# load df
df = pd.read_csv('../data/final_data.csv')

In [3]:
# orignal real data split into 70/15/15
train_size = int(len(df) * 0.7)
val_size = int(len(df) * 0.15)
val_end = train_size + val_size


train_data = df.iloc[:train_size].copy()
val_data = df.iloc[train_size:val_end].copy()
test_data = df.iloc[val_end:].copy()

# print length of data in each set
print(f'Train data length: {len(train_data)}')
print(f'Validation data length: {len(val_data)}')
print(f'Test data length: {len(test_data)}')

Train data length: 336
Validation data length: 72
Test data length: 72


# Reduce training set by 25%

In [4]:
# drop first 25% of original training period
train_start_idx2 = int(len(train_data) * 0.25)

train_df2 = train_data.iloc[train_start_idx2:].copy()
print("Reduced training length:", len(train_df2))
train_df2.head()

Reduced training length: 252


,Date,AUD_USD_ret,CAD_USD_ret,NZD_USD_ret,ZAR_USD_ret,CPI,TB3MS,M1,M2,wti_ret
84,1993-02-01,0.014696,-0.013935,0.006477,0.015980,-0.001409,-0.07,-0.002348,0.000290,0.054205
85,1993-03-01,0.035692,-0.010491,0.027203,0.018484,-0.000702,0.02,0.001726,0.000526,0.011383
86,1993-04-01,0.005345,0.011941,0.016422,-0.002258,0.002086,-0.08,0.004091,0.000703,-0.003451
87,1993-05-01,-0.018382,0.006122,0.007142,0.002185,-0.000705,0.09,0.008497,0.007594,-0.014926
88,1993-06-01,-0.034465,0.007155,-0.006308,0.019353,-0.002085,0.11,-0.008724,-0.005877,-0.044065


In [5]:
# grid search over hyperparameters for the expanding window LSTM
param_grid_tiny = {
    "lookback": [2, 10],
    "dropout": [0.001, 0.1],
    "units": [50, 170],
    "epochs": [50, 100]
}

param_combinations = list(itertools.product(
    param_grid_tiny["lookback"],
    param_grid_tiny["dropout"],
    param_grid_tiny["units"],
    param_grid_tiny["epochs"]
))

results_grid = []

for i, (lb, dr, units, ep) in enumerate(param_combinations, 1):
    print(f"\n[{i}/{len(param_combinations)}] Testing params:")
    print(f"lookback={lb}, dropout={dr}, units={units}, epochs={ep}")

    try:
        val_forecasts = expanding_window_lstm_forecast2(
            df=df,
            feature_cols=['wti_ret'],
            target_col='wti_ret',
            date_col="Date",
            train_start_idx=train_start_idx2,   
            initial_train_size=train_size,      # keep original validation start
            end_idx=val_end,                    # keep original validation end
            lookback=lb,
            units=units,
            dropout=dr,
            epochs=ep,
            batch_size=32,
            verbose=0,
            scale=True,
            seed=42
        )

        if len(val_forecasts) == 0:
            print("No forecasts generated, skipping.")
            continue

        mse = mean_squared_error(
            val_forecasts["actual"],
            val_forecasts["predicted"]
        )

        print(f"Validation MSE: {mse:.6f}")

        results_grid.append({
            "lookback": lb,
            "dropout": dr,
            "units": units,
            "epochs": ep,
            "mse": mse
        })

    except Exception as e:
        print(f"Error: {e}")
        continue

results_grid = pd.DataFrame(results_grid).sort_values("mse").reset_index(drop=True)


[1/16] Testing params:
lookback=2, dropout=0.001, units=50, epochs=50


2026-03-28 16:06:05.811435: W tensorflow/tsl/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


Error: Graph execution error:

Detected at node 'gradient_tape/mean_squared_error/Reshape_1' defined at (most recent call last):
    File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line 196, in _run_module_as_main
      return _run_code(code, main_globals, None,
    File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line 86, in _run_code
      exec(code, run_globals)
    File "/Users/jasonlow/Library/Python/3.10/lib/python/site-packages/ipykernel_launcher.py", line 17, in <module>
      app.launch_new_instance()
    File "/Users/jasonlow/Library/Python/3.10/lib/python/site-packages/traitlets/config/application.py", line 1075, in launch_instance
      app.start()
    File "/Users/jasonlow/Library/Python/3.10/lib/python/site-packages/ipykernel/kernelapp.py", line 739, in start
      self.io_loop.start()
    File "/Users/jasonlow/Library/Python/3.10/lib/python/site-packages/tornado/platform/asyncio.py", line 205, in start


2026-03-28 17:40:29.175361: W tensorflow/core/framework/op_kernel.cc:1830] OP_REQUIRES failed at resource_variable_ops.cc:611 : INVALID_ARGUMENT: Cannot update variable with shape [1] using a Tensor with shape [0], shapes must be equal.


Error: Graph execution error:

Detected at node 'AssignSubVariableOp_7' defined at (most recent call last):
    File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line 196, in _run_module_as_main
      return _run_code(code, main_globals, None,
    File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line 86, in _run_code
      exec(code, run_globals)
    File "/Users/jasonlow/Library/Python/3.10/lib/python/site-packages/ipykernel_launcher.py", line 17, in <module>
      app.launch_new_instance()
    File "/Users/jasonlow/Library/Python/3.10/lib/python/site-packages/traitlets/config/application.py", line 1075, in launch_instance
      app.start()
    File "/Users/jasonlow/Library/Python/3.10/lib/python/site-packages/ipykernel/kernelapp.py", line 739, in start
      self.io_loop.start()
    File "/Users/jasonlow/Library/Python/3.10/lib/python/site-packages/tornado/platform/asyncio.py", line 205, in start
      self.asyncio_lo

2026-03-28 17:57:39.782444: W tensorflow/core/framework/op_kernel.cc:1830] OP_REQUIRES failed at resource_variable_ops.cc:611 : INVALID_ARGUMENT: Cannot update variable with shape [] using a Tensor with shape [0], shapes must be equal.


In [6]:
results_df = pd.DataFrame(results_grid)
results_df = results_df.sort_values("mse")

print(results_df.head())

best_params = results_df.iloc[0]
print(best_params)

   lookback  dropout  units  epochs       mse
0         2    0.100     50      50  0.085461
1         2    0.100     50     100  0.085516
2         2    0.100    170      50  0.092133
3         2    0.100    170     100  0.092133
4         2    0.001    170     100  0.123901
lookback     2.000000
dropout      0.100000
units       50.000000
epochs      50.000000
mse          0.085461
Name: 0, dtype: float64


## Out of sample test

In [7]:
test_results = expanding_window_lstm_forecast2(
    df=df,
    feature_cols=['wti_ret'],
    target_col='wti_ret',
    train_start_idx=train_start_idx2,  
    initial_train_size=val_end,         # original test start
    end_idx=len(df),                    # forecast through end of sample
    date_col="Date",
    lookback=int(best_params["lookback"]),
    units=int(best_params["units"]),
    dropout=float(best_params["dropout"]),
    epochs=int(best_params["epochs"]),
    batch_size=32,
    verbose=0,
    scale=True,
    seed=42
)

test_mse = mean_squared_error(
    test_results["actual"],
    test_results["predicted"]
)

test_mape = mean_absolute_percentage_error(
    test_results["actual"],
    test_results["predicted"]
)

test_r2 = r2_score(
    test_results["actual"],
    test_results["predicted"]
)

print("Test MSE:", test_mse)
print("Test MAPE:", test_mape)
print("Test R²:", test_r2)

Test MSE: 0.23119621995433015
Test MAPE: 7.4948985865816855
Test R²: -11.201607251839608


In [8]:
# export test results to csv
test_results.to_csv('results/lstm_reduce25_results.csv', index=False)

# Reduce training data by 50%

In [9]:
# drop first 50% of original training period
train_start_idx3 = int(len(train_data) * 0.50)

train_df3 = train_data.iloc[train_start_idx3:].copy()
print("Reduced training length:", len(train_df3))
train_df3.head()


Reduced training length: 168


,Date,AUD_USD_ret,CAD_USD_ret,NZD_USD_ret,ZAR_USD_ret,CPI,TB3MS,M1,M2,wti_ret
168,2000-02-01,-0.044057,0.001800,-0.044693,0.030527,0.001168,0.23,-0.012015,-0.003236,0.074553
169,2000-03-01,-0.029704,0.006569,-0.000293,0.022928,0.001739,0.14,0.011111,0.003713,0.015876
170,2000-04-01,-0.022164,0.005530,0.011811,0.027526,-0.006450,-0.03,0.008280,0.005278,-0.148581
171,2000-05-01,-0.030088,0.018122,-0.052006,0.054991,0.002339,0.13,-0.016925,-0.014377,0.112759
172,2000-06-01,0.028260,-0.012575,-0.000782,-0.015663,0.004070,-0.10,0.007554,0.006298,0.100067


In [14]:
import gc
import tensorflow as tf
from tensorflow.keras import backend as K


In [19]:
# grid search over hyperparameters for the expanding window LSTM
param_grid_tiny = {
    "lookback": [2, 10],
    "dropout": [0.001, 0.1],
    "units": [50, 170],
    "epochs": [50, 100]
}

param_combinations = list(itertools.product(
    param_grid_tiny["lookback"],
    param_grid_tiny["dropout"],
    param_grid_tiny["units"],
    param_grid_tiny["epochs"]
))

results_grid3 = []

for i, (lb, dr, units, ep) in enumerate(param_combinations, 1):
    print(f"\n[{i}/{len(param_combinations)}] Testing params:")
    print(f"lookback={lb}, dropout={dr}, units={units}, epochs={ep}")

    try:
        # clear memory from previous run
        K.clear_session()
        tf.keras.backend.clear_session()
        gc.collect()

        val_forecasts = expanding_window_lstm_forecast2(
            df=df,
            feature_cols=['wti_ret'],
            target_col='wti_ret',
            date_col="Date",
            train_start_idx=train_start_idx3,
            initial_train_size=train_size,
            end_idx=val_end,
            lookback=lb,
            units=units,
            dropout=dr,
            epochs=ep,
            batch_size=32,
            verbose=0,
            scale=True,
            seed=42
        )

        if len(val_forecasts) == 0:
            print("No forecasts generated, skipping.")
            continue

        mse = mean_squared_error(
            val_forecasts["actual"],
            val_forecasts["predicted"]
        )

        print(f"Validation MSE: {mse:.6f}")

        results_grid3.append({
            "lookback": lb,
            "dropout": dr,
            "units": units,
            "epochs": ep,
            "mse": mse
        })

        
        del val_forecasts
        gc.collect()

    except Exception as e:
        print(f"Error: {e}")
        gc.collect()
        continue

results_grid3 = pd.DataFrame(results_grid3).sort_values("mse").reset_index(drop=True)


[1/16] Testing params:
lookback=2, dropout=0.001, units=50, epochs=50
Validation MSE: 0.129506

[2/16] Testing params:
lookback=2, dropout=0.001, units=50, epochs=100
Validation MSE: 0.140175

[3/16] Testing params:
lookback=2, dropout=0.001, units=170, epochs=50
Validation MSE: 0.112539

[4/16] Testing params:
lookback=2, dropout=0.001, units=170, epochs=100
Error: Graph execution error:

Detected at node 'mul_20' defined at (most recent call last):
    File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line 196, in _run_module_as_main
      return _run_code(code, main_globals, None,
    File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line 86, in _run_code
      exec(code, run_globals)
    File "/Users/jasonlow/Library/Python/3.10/lib/python/site-packages/ipykernel_launcher.py", line 17, in <module>
      app.launch_new_instance()
    File "/Users/jasonlow/Library/Python/3.10/lib/python/site-packages/traitlets/config/

2026-03-28 21:37:07.425693: W tensorflow/core/framework/op_kernel.cc:1830] OP_REQUIRES failed at resource_variable_ops.cc:611 : INVALID_ARGUMENT: Cannot update variable with shape [1] using a Tensor with shape [0], shapes must be equal.


Error: Graph execution error:

Detected at node 'AssignSubVariableOp_7' defined at (most recent call last):
    File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line 196, in _run_module_as_main
      return _run_code(code, main_globals, None,
    File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line 86, in _run_code
      exec(code, run_globals)
    File "/Users/jasonlow/Library/Python/3.10/lib/python/site-packages/ipykernel_launcher.py", line 17, in <module>
      app.launch_new_instance()
    File "/Users/jasonlow/Library/Python/3.10/lib/python/site-packages/traitlets/config/application.py", line 1075, in launch_instance
      app.start()
    File "/Users/jasonlow/Library/Python/3.10/lib/python/site-packages/ipykernel/kernelapp.py", line 739, in start
      self.io_loop.start()
    File "/Users/jasonlow/Library/Python/3.10/lib/python/site-packages/tornado/platform/asyncio.py", line 205, in start
      self.asyncio_lo

In [20]:
results_grid3

,lookback,dropout,units,epochs,mse
0,2,0.100,50,100,0.082530
1,2,0.100,50,50,0.082534
2,2,0.001,170,50,0.112539
3,2,0.100,170,50,0.118566
4,2,0.100,170,100,0.118566
5,2,0.001,50,50,0.129506
6,10,0.100,50,100,0.130078
7,10,0.100,50,50,0.130460
8,2,0.001,50,100,0.140175
9,10,0.100,170,50,0.198865


In [21]:
results_df3 = pd.DataFrame(results_grid3)
results_df3 = results_df3.sort_values("mse")

print(results_df3.head())

best_params = results_df3.iloc[0]
print(best_params)

   lookback  dropout  units  epochs       mse
0         2    0.100     50     100  0.082530
1         2    0.100     50      50  0.082534
2         2    0.001    170      50  0.112539
3         2    0.100    170      50  0.118566
4         2    0.100    170     100  0.118566
lookback      2.00000
dropout       0.10000
units        50.00000
epochs      100.00000
mse           0.08253
Name: 0, dtype: float64


### Out of sample test

In [22]:
test_results = expanding_window_lstm_forecast2(
    df=df,
    feature_cols=['wti_ret'],
    target_col='wti_ret',
    train_start_idx=train_start_idx3,  
    initial_train_size=val_end,         # original test start
    end_idx=len(df),                    # forecast through end of sample
    date_col="Date",
    lookback=int(best_params["lookback"]),
    units=int(best_params["units"]),
    dropout=float(best_params["dropout"]),
    epochs=int(best_params["epochs"]),
    batch_size=32,
    verbose=0,
    scale=True,
    seed=42
)

test_mse = mean_squared_error(
    test_results["actual"],
    test_results["predicted"]
)

test_mape = mean_absolute_percentage_error(
    test_results["actual"],
    test_results["predicted"]
)

test_r2 = r2_score(
    test_results["actual"],
    test_results["predicted"]
)

print("Test MSE:", test_mse)
print("Test MAPE:", test_mape)
print("Test R²:", test_r2)

Test MSE: 0.1823774361323279
Test MAPE: 8.044763969728548
Test R²: -8.62514805702146


In [23]:
# export test results to csv
test_results.to_csv('results/lstm_reduce50_results.csv', index=False)

# Reduce data by 75%

In [32]:
# drop first 75% of original training period
train_start_idx4 = int(len(train_data) * 0.75)

train_df4 = train_data.iloc[train_start_idx4:].copy()
print("Reduced training length:", len(train_df4))
train_df4.head()

Reduced training length: 84


,Date,AUD_USD_ret,CAD_USD_ret,NZD_USD_ret,ZAR_USD_ret,CPI,TB3MS,M1,M2,wti_ret
252,2007-02-01,0.000495,-0.004492,-0.001886,-0.001991,0.002213,0.05,-0.010088,-0.003136,0.083888
253,2007-03-01,0.012943,-0.002420,0.008751,0.024370,0.001316,-0.09,0.008927,0.002495,0.019379
254,2007-04-01,0.042161,-0.028809,0.048876,-0.034581,-0.002190,-0.07,0.005816,0.005323,0.056919
255,2007-05-01,-0.002306,-0.035787,-0.001489,-0.012447,0.001128,-0.14,-0.006278,-0.008145,-0.008161
256,2007-06-01,0.020303,-0.027760,0.031673,0.019302,-0.001810,-0.12,-0.011342,0.002648,0.061570


In [33]:
# grid search over hyperparameters for the expanding window LSTM
param_grid_tiny = {
    "lookback": [2, 10],
    "dropout": [0.001, 0.1],
    "units": [50, 170],
    "epochs": [50, 100]
}

param_combinations = list(itertools.product(
    param_grid_tiny["lookback"],
    param_grid_tiny["dropout"],
    param_grid_tiny["units"],
    param_grid_tiny["epochs"]
))

results_grid4 = []

for i, (lb, dr, units, ep) in enumerate(param_combinations, 1):
    print(f"\n[{i}/{len(param_combinations)}] Testing params:")
    print(f"lookback={lb}, dropout={dr}, units={units}, epochs={ep}")

    try:
        K.clear_session()
        tf.keras.backend.clear_session()
        gc.collect()
        
        val_forecasts = expanding_window_lstm_forecast2(
            df=df,
            feature_cols=['wti_ret'],
            target_col='wti_ret',
            date_col="Date",
            train_start_idx=train_start_idx4,   
            initial_train_size=train_size,      # keep original validation start
            end_idx=val_end,                    # keep original validation end
            lookback=lb,
            units=units,
            dropout=dr,
            epochs=ep,
            batch_size=32,
            verbose=0,
            scale=True,
            seed=42
        )

        if len(val_forecasts) == 0:
            print("No forecasts generated, skipping.")
            continue

        mse = mean_squared_error(
            val_forecasts["actual"],
            val_forecasts["predicted"]
        )

        print(f"Validation MSE: {mse:.6f}")

        results_grid4.append({
            "lookback": lb,
            "dropout": dr,
            "units": units,
            "epochs": ep,
            "mse": mse
        })

    except Exception as e:
        print(f"Error: {e}")
        continue

results_grid4 = pd.DataFrame(results_grid4).sort_values("mse").reset_index(drop=True)


[1/16] Testing params:
lookback=2, dropout=0.001, units=50, epochs=50
Validation MSE: 0.159270

[2/16] Testing params:
lookback=2, dropout=0.001, units=50, epochs=100
Validation MSE: 0.177103

[3/16] Testing params:
lookback=2, dropout=0.001, units=170, epochs=50
Validation MSE: 0.173262

[4/16] Testing params:
lookback=2, dropout=0.001, units=170, epochs=100
Validation MSE: 0.163020

[5/16] Testing params:
lookback=2, dropout=0.1, units=50, epochs=50
Validation MSE: 0.153779

[6/16] Testing params:
lookback=2, dropout=0.1, units=50, epochs=100
Validation MSE: 0.153355

[7/16] Testing params:
lookback=2, dropout=0.1, units=170, epochs=50
Validation MSE: 0.151165

[8/16] Testing params:
lookback=2, dropout=0.1, units=170, epochs=100
Validation MSE: 0.151203

[9/16] Testing params:
lookback=10, dropout=0.001, units=50, epochs=50
Validation MSE: 0.149317

[10/16] Testing params:
lookback=10, dropout=0.001, units=50, epochs=100
Validation MSE: 0.340588

[11/16] Testing params:
lookback=10

In [35]:
results_df4 = pd.DataFrame(results_grid4)
results_df4 = results_df4.sort_values("mse")

print(results_df4.head())

best_params = results_df4.iloc[0]
print(best_params)

   lookback  dropout  units  epochs       mse
0        10    0.100     50     100  0.112129
1        10    0.001     50      50  0.149317
2         2    0.100    170      50  0.151165
3         2    0.100    170     100  0.151203
4         2    0.100     50     100  0.153355
lookback     10.000000
dropout       0.100000
units        50.000000
epochs      100.000000
mse           0.112129
Name: 0, dtype: float64


# Out of sample test

In [37]:
test_results = expanding_window_lstm_forecast2(
    df=df,
    feature_cols=['wti_ret'],
    target_col='wti_ret',
    train_start_idx=train_start_idx4,  
    initial_train_size=val_end,         # original test start
    end_idx=len(df),                    # forecast through end of sample
    date_col="Date",
    lookback=int(best_params["lookback"]),
    units=int(best_params["units"]),
    dropout=float(best_params["dropout"]),
    epochs=int(best_params["epochs"]),
    batch_size=32,
    verbose=0,
    scale=True,
    seed=42
)

test_mse = mean_squared_error(
    test_results["actual"],
    test_results["predicted"]
)

test_mape = mean_absolute_percentage_error(
    test_results["actual"],
    test_results["predicted"]
)

test_r2 = r2_score(
    test_results["actual"],
    test_results["predicted"]
)

print("Test MSE:", test_mse)
print("Test MAPE:", test_mape)
print("Test R²:", test_r2)

In [ ]:
# export test results to csv
test_results.to_csv('results/lstm_reduce75_results.csv', index=False)